### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [29]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(row):
    # TODO: распарсить строку и вернуть dict перевода
    # TODO: при ошибке конвертации amount использовать raise ... from ...
    if len(row.split("|")) != 4:
        raise TransferFormatError("Строка должна состоять из 4 элементов")
    
    transfer_id, from_user, to_user, amount = row.split("|")
    try:
        amount = float(amount)
    except ValueError as e:
        raise TransferAmountError("Сумма перевода должна быть числом") from e
    
    if amount < 0:
        raise TransferAmountError("Сумма перевода должна быть положительной")
    
    return {
        "transfer_id": transfer_id,
        "from_user": from_user,
        "to_user": to_user,
        "amount": amount
    }


def apply_transfer(transfer, wallets):
    # TODO: проверить существование аккаунтов
    if transfer["from_user"] not in wallets:
        raise AccountNotFoundError("Отправитель не найден")
    if transfer["to_user"] not in wallets:
        raise AccountNotFoundError("Получатель не найден")
    # TODO: запретить перевод самому себе
    if transfer["from_user"] == transfer["to_user"]:
        raise TransferError("Перевод самому себе заперщен")
    # TODO: проверить совпадение валют
    from_wallet = wallets[transfer["from_user"]]
    to_wallet = wallets[transfer["to_user"]]
    if from_wallet["currency"] != to_wallet["currency"]:
        raise CurrencyMismatchError("Валюта отправителя не соответствует валюте получателя")
    # TODO: проверить баланс отправителя
    if from_wallet["balance"] < transfer["amount"]:
        raise InsufficientFundsError("Недостаточно средств на счете для перевода")
    # TODO: обновить балансы и вернуть dict результата
    from_wallet["balance"] -= transfer["amount"]
    to_wallet["balance"] += transfer["amount"]

    return {
        "transfer_id": transfer["transfer_id"],
        "from_user": transfer["from_user"],
        "to_user": transfer["to_user"],
        "amount": transfer["amount"],
        "currency": from_wallet["currency"]
    }


def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []
    for row in rows:
        try:
            transfer = parse_transfer(row)
            successes.append(apply_transfer(transfer, wallets))
        except TransferError as e:
            errors.append((row, type(e).__name__, e))
    return successes, errors


# TODO: вызвать process_batch(rows, wallets)
successes, errors = process_batch(rows, wallets)
# TODO: вывести успешные переводы
print(f"Успешные переводы: {len(successes)}")
for success in successes:
    print(success)
# TODO: вывести ошибки по типам
print(f"Ошибки: {len(errors)}")
type_errors = {}
for row, error, name in errors:
    if error not in type_errors:
        type_errors[error] = type_errors.get(error, 0) + 1
    print(f"Ошибка: '{error}', сообщение: '{name}', строка: '{row}'")
for error, count in type_errors.items():
    print(f"Ошибка '{error}' встречается {count} раз.")
# TODO: вывести итоговые балансы
for user, balance in wallets.items():
    print(f"Итоговый баланс {user} = {balance["balance"]} {balance["currency"]}")
# TODO: найти richest_usd_user
max_user = None
max_money = 0
for user, money in wallets.items():
    if money["currency"] == "USD":
        if money["balance"] > max_money:
            max_user = user
            max_money = money["balance"]
print(f"У пользовотеля {max_user} максимальный баланс = {max_money} USD")


Успешные переводы: 2
{'transfer_id': 'TR-100', 'from_user': 'alice', 'to_user': 'bob', 'amount': 200.0, 'currency': 'USD'}
{'transfer_id': 'TR-106', 'from_user': 'bob', 'to_user': 'dave', 'amount': 100.0, 'currency': 'USD'}
Ошибки: 5
Ошибка: 'InsufficientFundsError', сообщение: 'Недостаточно средств на счете для перевода', строка: 'TR-101|bob|dave|700'
Ошибка: 'CurrencyMismatchError', сообщение: 'Валюта отправителя не соответствует валюте получателя', строка: 'TR-102|alice|carol|50'
Ошибка: 'AccountNotFoundError', сообщение: 'Отправитель не найден', строка: 'TR-103|eve|bob|30'
Ошибка: 'TransferError', сообщение: 'Перевод самому себе заперщен', строка: 'TR-104|dave|dave|10'
Ошибка: 'TransferAmountError', сообщение: 'Сумма перевода должна быть числом', строка: 'TR-105|bob|alice|abc'
Ошибка 'InsufficientFundsError' встречается 1 раз.
Ошибка 'CurrencyMismatchError' встречается 1 раз.
Ошибка 'AccountNotFoundError' встречается 1 раз.
Ошибка 'TransferError' встречается 1 раз.
Ошибка 'Transfer